# Recommendation Systems
## CIS 3200 Final Project
### Charlie Argust, Kathryn Bigelow, Arbri Cenolli

In this project, we examine the best ways to recommend movies to a single user.

Data: https://www.kaggle.com/datasets/grouplens/movielens-20m-dataset?select=movie.csv

# Set Up

In [ ]:
# Load libraries
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
import math

In [ ]:
# Load data
movies = pd.read_csv("movie.csv")
ratings = pd.read_csv("rating.csv")

# Data Exploration and Filtering

Here, we take a subset of the data to work with so computations are easier on our machines.


In [ ]:
# We take a subset of 5000 users
unique_users = ratings['userId'].unique()[:5000]
ratings_subset = ratings[ratings['userId'].isin(unique_users)]
ratings_subset

,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40
...,...,...,...,...
750507,5000,800,4.0,1996-11-17 09:49:34
750508,5000,891,3.0,1996-11-17 09:54:23
750509,5000,1047,5.0,1996-11-17 09:55:34
750510,5000,1073,5.0,1996-11-17 09:45:26


In [ ]:
# We only keep movies that were seen by one of the 5000 users
unique_movies = ratings_subset['movieId'].unique()
movies_subset = movies[movies['movieId'].isin(unique_movies)]
movies_subset

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
27059,130052,Clown (2014),Drama|Horror
27069,130073,Cinderella (2015),Adventure|Children|Drama|Sci-Fi
27078,130219,The Dark Knight (2011),Action|Crime|Drama|Thriller
27122,130490,Insurgent (2015),Action|Romance|Sci-Fi


In [ ]:
# Merge the datasets so we have userId, movieId, title, rating all in one place
ratings_with_movies = pd.merge(ratings_subset, movies_subset[['movieId', 'title', 'genres']], on='movieId').drop("timestamp", axis=1)
ratings_with_movies

,userId,movieId,rating,title,genres
0,1,2,3.5,Jumanji (1995),Adventure|Children|Fantasy
1,5,2,3.0,Jumanji (1995),Adventure|Children|Fantasy
2,13,2,3.0,Jumanji (1995),Adventure|Children|Fantasy
3,29,2,3.0,Jumanji (1995),Adventure|Children|Fantasy
4,34,2,3.0,Jumanji (1995),Adventure|Children|Fantasy
...,...,...,...,...,...
750507,4967,109576,3.5,Welcome to the Jungle (2013),Comedy
750508,4967,112460,3.5,Planes: Fire & Rescue (2014),Adventure|Animation|Comedy
750509,4967,117871,3.5,The Water Diviner (2014),Action|Drama|War
750510,4989,399,3.0,Girl in the Cadillac (1995),Drama


# Non Personalized Recommendations

## Frequency

We could recommend movies that are simply rated most often in our dataset.

In [ ]:
movie_popularity = ratings_with_movies["title"].value_counts().sort_values(ascending=False)
movie_popularity[0:5]

title
Pulp Fiction (1994)                 2482
Forrest Gump (1994)                 2472
Silence of the Lambs, The (1991)    2294
Shawshank Redemption, The (1994)    2280
Jurassic Park (1993)                2211
Name: count, dtype: int64

If we were to recommend the 5 most popular movies to users in terms of frequency, we would recommed Pulp Fiction, Forrest Gump, The Silence of the Lambs, The Shawshank Redemption, and Jurassic Park.

## Average Rating

Just recommending frequency does not take into account how well the user liked the movies, though. So, we can weigh each film by their average rating.

In [ ]:
# We filter out movies with low frequencies
popular_movies = movie_popularity[movie_popularity > 100].index
popular_movies_rankings = ratings_with_movies[ratings_with_movies["title"].isin(popular_movies)]

# Get the average ratings for each of the popular movies
popular_movies_average_rankings = popular_movies_rankings[["title", "rating"]].groupby('title').mean().sort_values(by="rating", ascending=False)
popular_movies_average_rankings[0:5]

,rating
title,
"Shawshank Redemption, The (1994)",4.471711
"Godfather, The (1972)",4.382609
"Usual Suspects, The (1995)",4.374858
Band of Brothers (2001),4.356707
City of God (Cidade de Deus) (2002),4.294053


The movies with the highest average ratings are The Shawshank Redemption, The Godfather, The Usual Suspects, Band of Brothers, and City of God.

While it is simple to make recommendations based on simple aggregates like frequency and average ratings, these recommendations are not tailored to  users' specific interests. Although movies like The Shawshank redemption and The Godfather may be well liked by a wide variety of users, recommending these movies will not be very helpful if a users is looking to find a movie that fits their unique tastes.

# Personalized Recommendations

## Collaborative Filtering

Collaborative filtering will allow us to make recommendations to users based on how similar they rate movies to other users. This method will allow us to use the judgement of other users to find movies that our target will enjoy.

### Utility Matrix and Normalized Utility Matrix

First, we must set up our utility matrix of our users (rows) and movies (columns). In each cell is a rating of 1-5 given by a user for a movie they have seen. If the user has not seen the movie, then the cell contains NA.

We will normalize the utility matrix using each user's mean rating. This will center the ratings so that movies that have not been rated by the user will be treated as average.

In [ ]:
# Regular Utility matrix -- rows = users, cols = movies
utility_matrix = pd.pivot_table(ratings_subset, values="rating", index="userId", columns="movieId")

# Normalized by subtracting the mean of each user's ratings
norm_utility_matrix = utility_matrix.copy()
norm_utility_matrix = norm_utility_matrix.sub(norm_utility_matrix.mean(axis=1), axis=0)

# replace NAs with 0
norm_utility_matrix = norm_utility_matrix.fillna(0)

### Finding Similarity

In [ ]:
# Find the similarity between all users in the normalized utility matrix
similarity = cosine_similarity(norm_utility_matrix)
similarity_df = pd.DataFrame(similarity, index = norm_utility_matrix.index, columns = norm_utility_matrix.index)


In [ ]:
def _find_similar_users(userId, n_similar, df = similarity_df):
  similar_users = {}
  # return the userId's row
  # sorts the cosine similarities and returns the first n most similar

  similar_users_score = df.loc[userId].sort_values(ascending = False)[1 : n_similar + 1]

  for index, value in similar_users_score.items():
    similar_users[index] = value

  return similar_users

### Make Recommendations

In [ ]:
# Functions used to make predicted scores

def _find_user_watched_movies(userID, matrix):
    user_movies = matrix.loc[userID]
    watched_movies = user_movies[~np.isnan(user_movies.values)]
    return watched_movies.index

def _predict_scores(columns, pt, cosine_sims):
  arr = np.zeros(len(columns))
  current = 0
  for row in pt:
    indexes = np.where(row > 0)[0]
    numerator = np.sum(row[indexes] * cosine_sims[indexes])
    denom = np.sum(cosine_sims[indexes])
    prediction = round(numerator / denom, 2)
    arr[current] = prediction
    current += 1

  dictionary = dict(zip(columns, arr))
  return dict(sorted(dictionary.items(), key=lambda item: item[1], reverse = True))

def recommend(userID, df = similarity_df):
  similar_users = _find_similar_users(userID, 10)  # how many similar users is the best?
  sim_users_indices = sorted(similar_users.keys())

  # create subset pivot table for this user
  # utility matrix has scores 1-5 and we only want the similar users in the subset
  subset_pivot_table = utility_matrix[utility_matrix.index.isin(sim_users_indices)].fillna(0)

  # remove movies that no one has watched and remove movies user has already watched
  watched = _find_user_watched_movies(userID, utility_matrix)
  condition = subset_pivot_table.mean()       # Finds the movies w/ 0 mean rating (no one watched them)
  filtered_cols = condition[condition > 0]    # removes movies that no one has watched
  sim_users_pt = subset_pivot_table[filtered_cols.index]
  sim_users_pt = sim_users_pt[sim_users_pt.columns.difference(watched)]

  # predict scores
  np_matrix = sim_users_pt.values.T
  user_sim_scores = similarity_df.loc[userID]
  cosine_sims = user_sim_scores[user_sim_scores.index.isin(sim_users_indices)].values

  predicted_scores = _predict_scores(sim_users_pt.columns, np_matrix, cosine_sims)

  slice_dict = {x: predicted_scores[x] for x in list(predicted_scores)}

  return slice_dict

In [ ]:
def get_movie_titles(movieIds, movies):
    subset = movies[movies["movieId"].isin(movieIds)]
    return subset["title"].to_list()


We will get the top 5 movies for all users in our subset and write them to a .txt file.

In [ ]:
for user in ratings_subset["userId"].unique():
  scores = recommend(user)
  titles = get_movie_titles(list(scores.keys())[0:5], movies_subset)
  with open("Recommendation.txt", "a") as f:
    f.write(f"{user}" + "\t" + ",".join(titles) + "\n")


<ipython-input-17-3532875658f0>:15: RuntimeWarning: invalid value encountered in scalar divide
  prediction = round(numerator / denom, 2)


### Testing Collaborative Filtering

To test our recommendation system, we take a sample of 10 users and get an average of the root mean squared error between the users' actual ratings and the ratings we predict for them.

In [ ]:
# These users have movies that are hidden and thus can be tested for errors
users_to_test = [1, 145, 4999, 982, 3301, 2810, 16, 235, 2054, 1299]
rmse = []

for userId in users_to_test:

  utility_matrix = pd.pivot_table(ratings_subset, values="rating", index="userId", columns="movieId")

  # Fingure out how many movies to hide
  test_movies = round(len(utility_matrix.iloc[userId, :]) * .2)

  # Get just user U's row and save it for later
  original_user_movies = utility_matrix.loc[[userId]]

  # Get the indices of the movies we are hiding, so we can test them later
  hidden_movies = utility_matrix.loc[userId, 0:test_movies]
  hidden_movies = hidden_movies[hidden_movies.notna()]

  # Hide the first 20% of movies from the (convert them to np.nan)
  utility_matrix.loc[userId, 0:test_movies] = np.nan

  # Normalized Utility matrix
  norm_utility_matrix = utility_matrix.copy()
  norm_utility_matrix = norm_utility_matrix.sub(norm_utility_matrix.mean(axis=1), axis=0).fillna(0)

  # Use Cosine similarity on the normalized matrix --> gives us Pearson Coeffiecient
  similarity = cosine_similarity(norm_utility_matrix)
  similarity_df = pd.DataFrame(similarity, index = norm_utility_matrix.index, columns = norm_utility_matrix.index)

  # Get Predicitons
  test_predictions = recommend(userId)
  test_predictions = pd.Series(test_predictions).rename_axis("movieId", axis=0)

  # Find join the matricies so that it only keeps the hidden movies and removes movies that dont have a prediction
  movie_Ids = hidden_movies.index
  testing_matrix = pd.DataFrame(movie_Ids)
  testing_matrix = testing_matrix.merge(hidden_movies, how="left", left_on="movieId", right_on="movieId").rename(columns={userId:"actual_rating"})
  testing_matrix = testing_matrix.merge(test_predictions.rename("predicted_rating"), how="left", left_on="movieId", right_on="movieId")
  testing_matrix = testing_matrix.dropna(subset=["predicted_rating"])

  # Get the root mean squared error of the actual and predicted ratings
  error = mean_squared_error(testing_matrix["actual_rating"], testing_matrix["predicted_rating"], squared=False)
  rmse.append(error)

average_rmse = sum(rmse) / len(rmse)
average_rmse

<ipython-input-17-3532875658f0>:15: RuntimeWarning: invalid value encountered in scalar divide
  prediction = round(numerator / denom, 2)
<ipython-input-17-3532875658f0>:15: RuntimeWarning: invalid value encountered in scalar divide
  prediction = round(numerator / denom, 2)
<ipython-input-17-3532875658f0>:15: RuntimeWarning: invalid value encountered in scalar divide
  prediction = round(numerator / denom, 2)
<ipython-input-17-3532875658f0>:15: RuntimeWarning: invalid value encountered in scalar divide
  prediction = round(numerator / denom, 2)
<ipython-input-17-3532875658f0>:15: RuntimeWarning: invalid value encountered in scalar divide
  prediction = round(numerator / denom, 2)


0.9280321109545622

For the average RMSE of our sample, we get a RMSE of 0.93. This means that, on average, our predictions are off by 0.93 stars.

### Caveats for our Collaborative Filtering System

* We pick recommendations from the top scores in movieId order. This means that if there are multiple predicted 5 star ratings, we pick the 5 with the lowest movieId for simplicity.
* Since the utility matrix is so sparse, it is difficult to find movies that have been watched by more than one or two similar users. This means that predicted ratings may only be based on a few related users' input.


### Improvements for the Future

* We could further tailor the recommendations by taking the high predicted user-user scores and using collaborative filtering to find movies that most match the user's user profile.

* Ideally, we should also aquire more ratings data from the user so that the utility matrix is not so sparse. We may also consider using other data, including implicit user data, like clicks, or movie features, like genres or actors.



